In [ ]:
# ============================================================================
# SCRIPT UNIFICADO: Petrecca FFT Stacking, Bending Power-Law Fit & F_var
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from astropy.timeseries import LombScargle
import warnings

warnings.filterwarnings('ignore')

# ---------------------- 1. CONFIGURAÇÕES INICIAIS ----------------------
base_dir = r"ztffp_final_data_2025" 
csv_meta_path = r"Cross_15_luiz.csv"
band = 'g' # Focando na banda g para fugir da contaminação das linhas de emissão
plots_dir_base = "plots_Petrecca_Analysis"
os.makedirs(plots_dir_base, exist_ok=True)

# Parâmetros de Grid e Frequência
MIN_CADENCE_DAYS = 1.0 
TARGET_BINS = 500 
f_max_nyquist = 1.0 / (2.0 * MIN_CADENCE_DAYS) 

mass_edges = [7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]
edd_edges  = [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0]

# Função Bending Power-Law (Petrecca et al.)
def log_bending_power_law(f, logA, log_f_break, alpha, logC):
    f_break = 10**log_f_break
    A = 10**logA
    C = 10**logC
    P = A / (1.0 + (f / f_break)**alpha) + C
    return np.log10(P)

# ---------------------- 2. LEITURA E PROCESSAMENTO ----------------------
print("Lendo metadados e processando curvas de luz...")
meta = pd.read_csv(csv_meta_path.replace("\\", "/"))
ra_cat, dec_cat = meta['RA'].values, meta['DEC'].values
logmbh_cat, logedd_cat = meta['LOGMBH'].values, meta['LOGLEDD_RATIO'].values

folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and "plots" not in f]
valid_data = []

for folder in tqdm(folders, desc="Calculando F_var e Interpolando"):
    try: ra_f, dec_f = map(float, folder.split('_'))
    except: continue
    
    dist = np.sqrt((ra_cat - ra_f)**2 + (dec_cat - dec_f)**2)
    idx = np.argmin(dist)
    if dist[idx] > 1e-4: continue
    
    m, e = logmbh_cat[idx], logedd_cat[idx]
    if np.isnan(m) or np.isnan(e): continue
    if m < mass_edges[0] or m >= mass_edges[-1]: continue
    if e < edd_edges[0] or e >= edd_edges[-1]: continue

    txt_files = glob.glob(os.path.join(base_dir, folder, f"*_ztf_{band}_filtered.txt"))
    if not txt_files: continue
    
    try:
        # Lê a curva e tira a média diária
        df = pd.read_csv(txt_files[0])
        if len(df) < 10 or 'flux_dens' not in df.columns: continue
        
        df['mjd_int'] = np.floor(df['mjd']).astype(int)
        df_daily = df.groupby('mjd_int').agg({'mjd':'mean', 'flux_dens':'mean', 'flux_dens_err':'mean'}).reset_index()
        if len(df_daily) < 10: continue
        
        t = df_daily['mjd'].values
        flux = df_daily['flux_dens'].values
        flux_err = df_daily['flux_dens_err'].values
        
        # --- Cálculo Rigoroso do F_var ---
        S2 = np.var(flux, ddof=1)
        err2 = np.mean(flux_err**2)
        if S2 > err2:
            f_var = np.sqrt(S2 - err2) / np.mean(flux)
        else:
            f_var = np.nan # Variabilidade não detectada acima do ruído
            
    except: continue
    
    T = t.max() - t.min()
    valid_data.append({
        't': t, 'flux': flux, 'T': T, 'f_var': f_var,
        'LOGMBH': m, 'LOGLEDD_RATIO': e
    })

if not valid_data:
    raise ValueError("Nenhum dado válido encontrado. Verifique os caminhos.")

# Define Grid Global de Frequências
tau_max = max([d['T'] for d in valid_data])
global_fmin = 1.0 / tau_max
BIN_WIDTH = (f_max_nyquist - global_fmin) / TARGET_BINS
N_bins = int(np.ceil((f_max_nyquist - global_fmin) / BIN_WIDTH))
bin_edges = global_fmin + np.arange(N_bins + 1) * BIN_WIDTH
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

binned_matrix_ls = np.full((len(valid_data), N_bins), np.nan)
binned_matrix_fft = np.full((len(valid_data), N_bins), np.nan)

# ---------------------- 3. PSD: LOMB-SCARGLE VS PETRECCA (FFT) ----------------------
for i, d in enumerate(tqdm(valid_data, desc="Calculando LS e FFT")):
    t_raw, flux_raw = d['t'], d['flux']
    y_norm = (flux_raw - np.median(flux_raw)) / np.median(flux_raw)
    
    # LS Tradicional
    ls = LombScargle(t_raw - t_raw.min(), y_norm)
    f_ls, p_ls = ls.autopower(normalization='psd', maximum_frequency=f_max_nyquist)
    
    # Petrecca FFT
    T_total = t_raw.max() - t_raw.min()
    N_pts = int(np.ceil(T_total / MIN_CADENCE_DAYS))
    t_reg = np.linspace(t_raw.min(), t_raw.max(), N_pts)
    
    y_intp = interp1d(t_raw, y_norm, kind='linear', bounds_error=False, fill_value="extrapolate")(t_reg)
    fft_res = np.fft.rfft(y_intp)
    f_fft = np.fft.rfftfreq(N_pts, d=MIN_CADENCE_DAYS)[1:]
    p_fft = ((np.abs(fft_res)**2) * (2.0 * MIN_CADENCE_DAYS / N_pts))[1:]
    
    # Binning Simulado
    for f_arr, p_arr, out_matrix in zip([f_ls, f_fft], [p_ls, p_fft], [binned_matrix_ls, binned_matrix_fft]):
        inds = np.searchsorted(bin_edges, f_arr, side='right') - 1
        valid_mask = (inds >= 0) & (inds < N_bins)
        if np.any(valid_mask):
            bid, pw = inds[valid_mask], p_arr[valid_mask]
            sums = np.bincount(bid, weights=pw, minlength=N_bins)
            counts = np.bincount(bid, minlength=N_bins)
            mask = counts > 0
            out_matrix[i, mask] = sums[mask] / counts[mask]

# ---------------------- 4. PLOT 1: VAZAMENTO DA FUNÇÃO JANELA ----------------------
print("\nGerando Plot de Comparação de Leakage...")
med_ls = np.nanmedian(binned_matrix_ls, axis=0)
med_fft = np.nanmedian(binned_matrix_fft, axis=0)

plt.figure(figsize=(9, 6))
plt.plot(np.log10(bin_centers), np.log10(med_ls), label='Lomb-Scargle (Com Buracos)', color='red', alpha=0.8, lw=2)
plt.plot(np.log10(bin_centers), np.log10(med_fft), label='Petrecca Method (Interpolação + FFT)', color='blue', alpha=0.8, lw=2)
plt.xlabel("log10(Frequência [dia$^{-1}$])", fontweight='bold')
plt.ylabel("log10(Power)", fontweight='bold')
plt.title(f"Red-Noise Leakage: Lomb-Scargle vs FFT (Banda {band})", fontweight='bold')
plt.legend()
plt.grid(True, ls=':', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir_base, "leakage_comparison.png"), dpi=200)
plt.close()

# ---------------------- 5. PLOT 2: AJUSTE DO BENDING POWER-LAW POR tau_max ----------------------
print("Ajustando o Bending Power-Law e plotando Grid...")
LOGMBH_arr = np.array([d['LOGMBH'] for d in valid_data])
LOGLEDD_arr = np.array([d['LOGLEDD_RATIO'] for d in valid_data])

mi_array = np.digitize(LOGMBH_arr, mass_edges) - 1
ej_array = np.digitize(LOGLEDD_arr, edd_edges) - 1
T_track_array = ej_array - mi_array

valid_grid = (mi_array >= 0) & (mi_array < len(mass_edges)-1) & (ej_array >= 0) & (ej_array < len(edd_edges)-1)
unique_tracks = np.unique(T_track_array[valid_grid])

cols = 3
rows = int(np.ceil(len(unique_tracks) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows), sharex=True, sharey=True)
axes = axes.flatten()

for idx, track in enumerate(sorted(unique_tracks)):
    ax = axes[idx]
    obj_idxs = np.where(valid_grid & (T_track_array == track))[0]
    
    if len(obj_idxs) < 2: 
        ax.set_visible(False)
        continue
    
    # Extrai PSD via FFT para fit limpo
    arr = binned_matrix_fft[obj_idxs, :] 
    good_mask = np.sum(~np.isnan(arr), axis=0) >= 1
    med_psd = np.full(N_bins, np.nan)
    med_psd[good_mask] = np.nanmedian(arr[:, good_mask], axis=0)
    
    f_fit = bin_centers[good_mask]
    p_fit = med_psd[good_mask]
    valid = (~np.isnan(f_fit)) & (~np.isnan(p_fit)) & (p_fit > 0)
    f_fit, log_p_fit = f_fit[valid], np.log10(p_fit[valid])
    
    ax.scatter(np.log10(f_fit), log_p_fit, color='k', s=10, alpha=0.5, label='Median FFT PSD')
    
    if len(f_fit) > 10:
        p0 = [np.max(log_p_fit), -3.0, 2.0, np.min(log_p_fit)]
        bounds = ([-10.0, -4.5, 0.0, -10.0], [10.0, -1.0, 4.0, 10.0])
        try:
            popt, _ = curve_fit(log_bending_power_law, f_fit, log_p_fit, p0=p0, bounds=bounds)
            logA, log_f_break, alpha, logC = popt
            
            f_smooth = np.logspace(np.log10(f_fit.min()), np.log10(f_fit.max()), 100)
            log_p_smooth = log_bending_power_law(f_smooth, *popt)
            
            ax.plot(np.log10(f_smooth), log_p_smooth, color='red', lw=2)
            ax.axvline(log_f_break, color='blue', ls='--', lw=1.5)
            
            # Anotações: Alpha e F_var médio daquele bin
            mean_fvar = np.nanmean([valid_data[k]['f_var'] for k in obj_idxs])
            txt = f"$\\alpha={alpha:.2f}$\n$\log f_b={log_f_break:.2f}$\n$\\bar{{F}}_{{var}}={mean_fvar:.3f}$"
            ax.text(0.60, 0.70, txt, transform=ax.transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
        except:
            ax.text(0.5, 0.5, "Fit Failed", transform=ax.transAxes, ha='center')
            
    ratio_val = -9.5 + track * 0.5
    ax.set_title(f"$\log(\lambda/M) \approx {ratio_val:.1f}$ (N={len(obj_idxs)})")
    ax.grid(ls=':', alpha=0.4)

for ax in axes[-cols:]: ax.set_xlabel(r"$\log_{10}(Frequency \ [day^{-1}])$", fontweight='bold')
for ax in axes[::cols]: ax.set_ylabel(r"$\log_{10}(Power)$", fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(plots_dir_base, "bending_pl_grid.png"), dpi=200)
plt.show()

print("\nProcessamento Finalizado com Sucesso! Gráficos salvos na pasta:", plots_dir_base)

Lendo metadados e processando curvas de luz...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\vitma\\Documents\\GitHub\\AGN-Power-Spectra\\ztffp_final_data_2025'